# WattWise — Phase 2: Looking at the data

**What this notebook is for:** make a few charts from the real data we collected in Phase 1, and answer the one question WattWise is built on:

> **Are the cheapest hours also the greenest?**

Run each cell in order (click it, then press **Shift + Enter**). Plain-English notes sit above every code cell, so you always know *why* you're running it.

*Reminder: this is real published data (Octopus Agile prices + NESO carbon), not simulated. Prices only exist up to where Octopus has published them — the rest is blank on purpose, and that gap is what Phase 3 forecasting will fill.*

## 1. Load the combined file

This reads `energy_data.csv` (the file `prepare_data.py` made) and shows the first rows. We also add a **UK local-time** column for the charts — the file stores UTC, and we only switch to local time when we *look* at it, so clocks changing never corrupts the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# load the file Phase 1 produced
df = pd.read_csv("../backend/data/processed/energy_data.csv", parse_dates=["timestamp_utc"])

# add a UK local-time column just for display
df["time_local"] = df["timestamp_utc"].dt.tz_convert("Europe/London")

print(f"{len(df)} half-hourly rows")
print(f"{df['price_p_kwh'].notna().sum()} of them have a published price")
df.head()

## 2. Chart 1 — electricity price across the day

Only the rows that actually have a published price. Watch for the **evening peak** (roughly 4–7pm) — that's when power is most expensive.

In [ ]:
priced = df.dropna(subset=["price_p_kwh"]).copy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(priced["time_local"], priced["price_p_kwh"], marker="o", color="#2563EB")
ax.set_title("Electricity price across the day — North West England (Agile)")
ax.set_ylabel("pence per kWh")
ax.set_xlabel("time of day")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

print(f"Cheapest: {priced['price_p_kwh'].min():.2f} p/kWh")
print(f"Dearest:  {priced['price_p_kwh'].max():.2f} p/kWh")

## 3. Chart 2 — grid carbon intensity across 48 hours

Carbon is published for the full next 48h, so this line is longer than the price line. Lower = cleaner electricity (more wind/solar, less gas).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["time_local"], df["carbon_intensity"], color="#10B981")
ax.set_title("Grid carbon intensity — next 48 hours")
ax.set_ylabel("gCO2 per kWh")
ax.set_xlabel("time")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

print(f"Greenest: {df['carbon_intensity'].min():.0f} gCO2/kWh")
print(f"Dirtiest: {df['carbon_intensity'].max():.0f} gCO2/kWh")

## 4. Do price and carbon move together?

If cheap always meant green, WattWise's job would be trivial. Let's check. We put price on one axis and carbon on the other, for every half-hour where we have both, and measure the **correlation** — a number from -1 to +1:

- near **+1** → when one is high the other is high (they agree)
- near **0** → no real relationship
- near **-1** → when one is high the other is low (they disagree)

In [ ]:
both = df.dropna(subset=["price_p_kwh"])
correlation = both["price_p_kwh"].corr(both["carbon_intensity"])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(both["carbon_intensity"], both["price_p_kwh"], color="#7C3AED")
ax.set_title(f"Price vs carbon  (correlation = {correlation:.2f})")
ax.set_xlabel("carbon intensity (gCO2/kWh)")
ax.set_ylabel("price (p/kWh)")
ax.grid(alpha=0.3)
plt.show()

print(f"Correlation between price and carbon: {correlation:.2f}")

## 5. Cheapest hours vs greenest hours

The real test: are the 3 cheapest slots the *same* slots as the 3 greenest? If they line up, "cheap" and "green" are the same goal. If they don't, WattWise genuinely has to **trade off** cost against carbon — which is the more interesting (and honest) story for your dissertation.

In [ ]:
cheapest = priced.nsmallest(3, "price_p_kwh")[["time_local", "price_p_kwh", "carbon_intensity"]]
greenest = priced.nsmallest(3, "carbon_intensity")[["time_local", "price_p_kwh", "carbon_intensity"]]

print("3 CHEAPEST slots:")
print(cheapest.to_string(index=False))
print("\n3 GREENEST slots (that also have a price):")
print(greenest.to_string(index=False))

# do they overlap?
overlap = set(cheapest["time_local"]) & set(greenest["time_local"])
print(f"\nSlots that are BOTH cheapest and greenest: {len(overlap)} out of 3")

## 6. Your finding ✍️

Write one or two plain sentences here about what you saw. A good finding looks like:

> *"In this 48-hour sample, the cheapest hours were late evening (~10–11pm), but the greenest hours were [...]. The correlation was [...], which means cost and carbon [do / don't] line up — so WattWise has to let the user choose what matters."*

**This written finding is the whole point of Phase 2.** Once it's here, Phase 2 is done ✅ — commit and push, then we start Phase 3 (forecasting).

*Note: with only one 48-hour snapshot, treat this as a first look. As you re-run the Phase 1 scripts on other days you'll gather more, and the pattern gets clearer.*